<a href="https://colab.research.google.com/github/NathanAndrewsBYU/Stylometry-Federalist-Papers/blob/main/notebooks/burrows_delta_balanced_n65.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Burrows' Delta: Balanced Leave-One-Out (65 known papers)

Runs the classic Burrows' Delta function-word baseline on the 65 known-authorship
Federalist Papers (51 Hamilton, 14 Madison), for comparison against the GPT-2
perplexity method (balanced runs, same 65-paper test pool).

**What this notebook does:**
1. Downloads the Federalist Papers from Project Gutenberg and sorts them into
   author folders.
2. Balances Hamilton down to 14 papers to match Madison (correcting the
   51-vs-14 imbalance that biases Delta toward Hamilton). The balanced profile
   is built from the subsample, but ALL 51 Hamilton + 14 Madison papers are
   evaluated (n = 65), matching the GPT-2 evaluation.
3. Repeats across several balance seeds so the result doesn't depend on which
   14 Hamilton papers get picked.
4. Reports overall, Hamilton and Madison accuracy per seed.

No GPU, no fine-tuning, runs in seconds.

## 1. Download and prepare the corpus

In [ ]:
import os
import re
import glob
import requests

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/1404/pg1404.txt"
CORPUS_ROOT = "corpus/federalist"

HAMILTON_SOLO = set([1] + list(range(6, 10)) + list(range(11, 14)) + list(range(15, 18)) +
                     list(range(21, 37)) + list(range(59, 62)) + list(range(65, 86)))
MADISON_SOLO = set([10, 14] + list(range(37, 49)))
JAY = set([2, 3, 4, 5, 64])
JOINT = set([18, 19, 20])
DISPUTED = set(list(range(49, 59)) + [62, 63])


def folder_for(number):
    if number in HAMILTON_SOLO:
        return "hamilton"
    if number in MADISON_SOLO:
        return "madison"
    if number in JAY:
        return "jay"
    if number in JOINT:
        return "joint"
    if number in DISPUTED:
        return "disputed"
    return None


def prepare_corpus():
    print("Downloading Federalist Papers from Project Gutenberg...")
    response = requests.get(GUTENBERG_URL, timeout=30)
    response.raise_for_status()
    raw_text = response.text
    print(f"Downloaded {len(raw_text):,} characters.")

    start_marker = "*** START OF"
    end_marker = "*** END OF"
    start_idx = raw_text.find(start_marker)
    end_idx = raw_text.find(end_marker)
    if start_idx != -1 and end_idx != -1:
        body_start = raw_text.find("\n", start_idx) + 1
        body_text = raw_text[body_start:end_idx]
    else:
        print("WARNING: could not find Gutenberg markers, using full text.")
        body_text = raw_text

    header_pattern = re.compile(r"FEDERALIST\.?\s+No\.?\s+(\d+)", re.IGNORECASE)
    matches = list(header_pattern.finditer(body_text))
    print(f"Found {len(matches)} header matches (expect at least 85).")

    papers = {}
    for i, match in enumerate(matches):
        number = int(match.group(1))
        section_start = match.end()
        section_end = matches[i + 1].start() if i + 1 < len(matches) else len(body_text)
        text = body_text[section_start:section_end].strip()
        if number not in papers or len(text) > len(papers[number]):
            papers[number] = text

    missing = sorted(set(range(1, 86)) - set(papers.keys()))
    if missing:
        print(f"WARNING: missing papers {missing}")

    assert len(HAMILTON_SOLO) == 51 and len(MADISON_SOLO) == 14
    assert len(JAY) == 5 and len(JOINT) == 3 and len(DISPUTED) == 12

    for folder in ["hamilton", "madison", "jay", "joint", "disputed"]:
        os.makedirs(os.path.join(CORPUS_ROOT, folder), exist_ok=True)

    written = 0
    for number, text in papers.items():
        folder = folder_for(number)
        if folder is None:
            continue
        out_path = os.path.join(CORPUS_ROOT, folder, f"federalist_{number:02d}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
        written += 1

    print(f"\nWrote {written} paper files into {CORPUS_ROOT}/")
    for folder in ["hamilton", "madison", "jay", "joint", "disputed"]:
        n = len(glob.glob(os.path.join(CORPUS_ROOT, folder, "*.txt")))
        print(f"  {folder}: {n} files")


prepare_corpus()

Downloaded 1,186,743 characters.
Found 85 header matches (expect at least 85).

Wrote 85 paper files into corpus/federalist/
  hamilton: 51 files
  madison: 14 files
  jay: 5 files
  joint: 3 files
  disputed: 12 files


## 2. Burrows' Delta implementation (balanced)

In [ ]:
import csv
import random
import statistics
from collections import Counter

N_FEATURES = 100


def list_txt_files(subfolder):
    return sorted(glob.glob(os.path.join(CORPUS_ROOT, subfolder, "*.txt")))


def clean_text(raw):
    text = raw
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"PUBLIUS\.?\s*$", "", text.strip())
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text):
    return re.findall(r"[a-z\']+", text.lower())


def load_tokens(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return tokenize(clean_text(f.read()))


def word_frequencies(tokens, vocabulary):
    total = len(tokens)
    counts = Counter(tokens)
    return {w: (counts.get(w, 0) / total) * 1000 for w in vocabulary}


def build_vocabulary(all_files, n_features):
    counts = Counter()
    for path in all_files:
        counts.update(load_tokens(path))
    return [w for w, _ in counts.most_common(n_features)]


def author_profile(files, vocabulary):
    freq_rows = [word_frequencies(load_tokens(f), vocabulary) for f in files]
    profile = {}
    for w in vocabulary:
        values = [row[w] for row in freq_rows]
        mean = statistics.mean(values)
        stdev = statistics.stdev(values) if len(values) > 1 else 0
        stdev = stdev if stdev > 0 else 1e-6
        profile[w] = (mean, stdev)
    return profile


def burrows_delta(paper_freqs, profile, vocabulary):
    z_scores = [abs((paper_freqs[w] - profile[w][0]) / profile[w][1]) for w in vocabulary]
    return sum(z_scores) / len(z_scores)


def balance_hamilton(hamilton_files, madison_files, balance_seed):
    """Subsample Hamilton\'s papers down to Madison\'s count, reproducibly."""
    if balance_seed is None or len(hamilton_files) <= len(madison_files):
        return hamilton_files
    rng = random.Random(balance_seed)
    return sorted(rng.sample(hamilton_files, len(madison_files)))


def leave_one_out_delta(hamilton_files, madison_files, n_features=N_FEATURES,
                         balance_seed=42, verbose=True):
    """
    Evaluates against ALL of hamilton_files, not just the balanced subsample
    -- matching leave_one_out_validation_balanced on the GPT-2 side
    (stylometry_lib.py), so accuracy numbers from the two methods are
    computed over the same-size test pool and are actually comparable.

    A "balanced full Hamilton" profile is built once from the subsample
    (balance_seed picks which papers) and reused whenever the held-out
    paper is Madison's, or a Hamilton paper NOT in the subsample (never
    included in the profile to begin with, so already validly held-out --
    no need to rebuild it). If the held-out paper IS one of the sampled
    Hamilton papers, a "balanced minus-one" profile is rebuilt excluding it.
    Madison's profile is always rebuilt minus whichever paper is held out,
    since every Madison paper is always in its training corpus.
    """
    balanced_hamilton = balance_hamilton(hamilton_files, madison_files, balance_seed)
    balanced_hamilton_set = set(balanced_hamilton)

    if verbose and balance_seed is not None and len(balanced_hamilton) < len(hamilton_files):
        print(f"Balancing: using {len(balanced_hamilton)}/{len(hamilton_files)} Hamilton "
              f"papers (seed={balance_seed}) vs. {len(madison_files)} Madison papers.")

    all_files = balanced_hamilton + madison_files
    vocabulary = build_vocabulary(all_files, n_features)
    if verbose:
        print(f"Using top {len(vocabulary)} most frequent words as features.")
        print(f"Most frequent 15: {vocabulary[:15]}")
        print(f"Evaluating against all {len(hamilton_files)} Hamilton papers "
              f"(not just the {len(balanced_hamilton)}-paper balanced subsample) "
              f"and all {len(madison_files)} Madison papers.")

    full_ham_profile = author_profile(balanced_hamilton, vocabulary)
    full_mad_profile = author_profile(madison_files, vocabulary)

    all_papers = [("hamilton", f) for f in hamilton_files] + [("madison", f) for f in madison_files]
    results = []

    for true_author, held_out_file in all_papers:
        if true_author == "hamilton":
            if held_out_file in balanced_hamilton_set:
                remaining = [f for f in balanced_hamilton if f != held_out_file]
                ham_profile = author_profile(remaining, vocabulary)
            else:
                ham_profile = full_ham_profile  # never in the subsample -- already valid
            mad_profile = full_mad_profile
        else:
            remaining_madison = [f for f in madison_files if f != held_out_file]
            mad_profile = author_profile(remaining_madison, vocabulary)
            ham_profile = full_ham_profile

        held_out_freqs = word_frequencies(load_tokens(held_out_file), vocabulary)
        delta_h = burrows_delta(held_out_freqs, ham_profile, vocabulary)
        delta_m = burrows_delta(held_out_freqs, mad_profile, vocabulary)

        predicted_author = "hamilton" if delta_h < delta_m else "madison"
        correct = predicted_author == true_author

        results.append({
            "file": os.path.basename(held_out_file),
            "true_author": true_author,
            "predicted_author": predicted_author,
            "delta_hamilton": delta_h,
            "delta_madison": delta_m,
            "correct": correct,
            "balance_seed": balance_seed,
        })
        if verbose:
            print(f"{os.path.basename(held_out_file)} | true={true_author} pred={predicted_author} "
                  f"(Delta H:{delta_h:.3f} M:{delta_m:.3f}) {'CORRECT' if correct else 'WRONG'}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    n_ham = sum(1 for r in results if r["true_author"] == "hamilton")
    n_mad = sum(1 for r in results if r["true_author"] == "madison")
    correct_ham = sum(1 for r in results if r["true_author"] == "hamilton" and r["correct"])
    correct_mad = sum(1 for r in results if r["true_author"] == "madison" and r["correct"])

    if verbose:
        print(f"\nOverall Burrows' Delta leave-one-out accuracy (balance_seed={balance_seed}): "
              f"{accuracy:.1%} ({sum(r['correct'] for r in results)}/{len(results)})")
        print(f"  Hamilton: {correct_ham}/{n_ham} ({correct_ham/n_ham:.1%})")
        print(f"  Madison: {correct_mad}/{n_mad} ({correct_mad/n_mad:.1%})")
    return results


def save_results_csv(results, out_path):
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"Saved {len(results)} results to {out_path}")

## 3. Run the balanced experiment (65 known papers)

Hamilton is balanced to 14 papers for the profile; all 65 known papers are
evaluated. Sweeps several balance seeds.

In [ ]:
BALANCE_SEEDS = [1, 2, 3, 4, 5]

hamilton_files = list_txt_files("hamilton")
madison_files = list_txt_files("madison")
print(f"Loaded {len(hamilton_files)} Hamilton, {len(madison_files)} Madison papers.\n")

all_seed_results = {}
combined_results = []
for seed in BALANCE_SEEDS:
    print(f"=== balance_seed={seed} ===")
    results = leave_one_out_delta(hamilton_files, madison_files, balance_seed=seed)
    all_seed_results[seed] = results
    combined_results.extend(results)
    print()

os.makedirs("results", exist_ok=True)
save_results_csv(combined_results, "results/traditional_burrows_delta_balanced.csv")

Loaded 51 Hamilton, 14 Madison papers.

=== balance_seed=1 ===
Balancing: using 14/51 Hamilton papers (seed=1) vs. 14 Madison papers.
Using top 100 most frequent words as features.
Most frequent 15: ['the', 'of', 'to', 'and', 'in', 'a', 'be', 'that', 'it', 'is', 'which', 'by', 'as', 'have', 'this']
Evaluating against all 51 Hamilton papers (not just the 14-paper balanced subsample) and all 14 Madison papers.
federalist_01.txt | true=hamilton pred=hamilton (Delta H:0.942 M:1.160) CORRECT
federalist_06.txt | true=hamilton pred=hamilton (Delta H:0.949 M:1.051) CORRECT
federalist_07.txt | true=hamilton pred=hamilton (Delta H:0.982 M:1.207) CORRECT
federalist_08.txt | true=hamilton pred=hamilton (Delta H:0.719 M:0.882) CORRECT
federalist_09.txt | true=hamilton pred=hamilton (Delta H:0.863 M:0.901) CORRECT
federalist_11.txt | true=hamilton pred=hamilton (Delta H:0.837 M:1.182) CORRECT
federalist_12.txt | true=hamilton pred=hamilton (Delta H:0.751 M:0.988) CORRECT
federalist_13.txt | true=ham

## 4. Summarize results

In [ ]:
import pandas as pd

summary_rows = []
for seed, results in all_seed_results.items():
    ham = [r for r in results if r["true_author"] == "hamilton"]
    mad = [r for r in results if r["true_author"] == "madison"]
    summary_rows.append({
        "balance_seed": seed,
        "overall_accuracy": sum(r["correct"] for r in results) / len(results),
        "hamilton_accuracy": sum(r["correct"] for r in ham) / len(ham),
        "madison_accuracy": sum(r["correct"] for r in mad) / len(mad),
        "n_total": len(results),
        "misclassified": ", ".join(sorted(r["file"][10:12] for r in results if not r["correct"])) or "none",
    })

summary_df = pd.DataFrame(summary_rows)
save_results_csv(summary_rows, "results/traditional_burrows_delta_balanced_summary.csv")
print(f"Mean overall accuracy across seeds: {summary_df.overall_accuracy.mean():.1%}")
summary_df

Saved 5 results to results/traditional_burrows_delta_balanced_summary.csv
Mean overall accuracy across seeds: 92.3%


,balance_seed,overall_accuracy,hamilton_accuracy,madison_accuracy,n_total,misclassified
0,1,0.938462,0.960784,0.857143,65,"_1, _2, _3, _8"
1,2,0.938462,1.000000,0.714286,65,"_1, _1, _3, _4"
2,3,0.923077,0.980392,0.714286,65,"_1, _1, _3, _4, _8"
3,4,0.907692,0.882353,1.000000,65,"_2, _2, _3, _8, _8, _8"
4,5,0.907692,1.000000,0.571429,65,"_1, _1, _3, _3, _4, _4"


## 5. Which papers get misclassified, and how often, across seeds

In [ ]:
df = pd.DataFrame(combined_results)
pivot = df.pivot_table(index=["file", "true_author"], columns="balance_seed", values="correct")
pivot["times_wrong"] = (1 - pivot).sum(axis=1)
pivot[pivot.times_wrong > 0].sort_values("times_wrong", ascending=False)

,balance_seed,1,2,3,4,5,times_wrong
file,true_author,,,,,,
federalist_14.txt,madison,0.0,0.0,0.0,1.0,0.0,4.0
federalist_38.txt,madison,0.0,0.0,0.0,1.0,0.0,4.0
federalist_10.txt,madison,1.0,0.0,0.0,1.0,0.0,3.0
federalist_80.txt,hamilton,0.0,1.0,0.0,0.0,1.0,3.0
federalist_23.txt,hamilton,0.0,1.0,1.0,0.0,1.0,2.0
federalist_44.txt,madison,1.0,0.0,1.0,1.0,0.0,2.0
federalist_28.txt,hamilton,1.0,1.0,1.0,0.0,1.0,1.0
federalist_33.txt,hamilton,1.0,1.0,1.0,0.0,1.0,1.0
federalist_39.txt,madison,1.0,1.0,1.0,1.0,0.0,1.0


## 6. Download results

In [ ]:
from google.colab import files

files.download("results/traditional_burrows_delta_balanced.csv")
files.download("results/traditional_burrows_delta_balanced_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>